In [1]:
import pandas as pd
import numpy as np

# STEP 1: Create sample messy e-commerce dataset (8 mins)
np.random.seed(42)
data = {
    'customer_id': [1, 2, 2, 3, 4, None, 5, 6, 7, 8, 9, 10] * 3,
    'purchase_amount': [100, 250, 250, None, 450, 500, 120, 999, 150, 160, 170, 5000] * 3,
    'region': ['North', 'South', 'South', 'East', 'West', 'North', None, 'East', 'South', 'West', 'North', 'East'] * 3,
    'days_to_ship': [5, 3, 3, 4, 2, 6, 7, 8, 4, 5, 3, 180] * 3
}
df = pd.DataFrame(data)
print("=" * 50)
print("ORIGINAL DATA (with issues)")
print("=" * 50)
print(df.head(10))
print(f"\nShape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")

ORIGINAL DATA (with issues)
   customer_id  purchase_amount region  days_to_ship
0          1.0            100.0  North             5
1          2.0            250.0  South             3
2          2.0            250.0  South             3
3          3.0              NaN   East             4
4          4.0            450.0   West             2
5          NaN            500.0  North             6
6          5.0            120.0   None             7
7          6.0            999.0   East             8
8          7.0            150.0  South             4
9          8.0            160.0   West             5

Shape: (36, 4)

Data types:
customer_id        float64
purchase_amount    float64
region              object
days_to_ship         int64
dtype: object


In [2]:
print("\n" + "=" * 50)
print("MISSING VALUE ANALYSIS")
print("=" * 50)
missing = df.isnull().sum()
print("\nMissing values per column:")
print(missing)
print(f"\nMissing % per column:")
print((missing / len(df) * 100).round(2))


MISSING VALUE ANALYSIS

Missing values per column:
customer_id        3
purchase_amount    3
region             3
days_to_ship       0
dtype: int64

Missing % per column:
customer_id        8.33
purchase_amount    8.33
region             8.33
days_to_ship       0.00
dtype: float64


In [3]:
print("\n" + "=" * 50)
print("DUPLICATE ANALYSIS")
print("=" * 50)
print(f"\nTotal duplicates (exact rows): {df.duplicated().sum()}")
print("\nDuplicate rows:")
print(df[df.duplicated(keep=False)].sort_values('customer_id'))

# Check duplicates on specific column
print(f"\nDuplicate customer_ids: {df['customer_id'].duplicated().sum()}")


DUPLICATE ANALYSIS

Total duplicates (exact rows): 25

Duplicate rows:
    customer_id  purchase_amount region  days_to_ship
0           1.0            100.0  North             5
12          1.0            100.0  North             5
24          1.0            100.0  North             5
2           2.0            250.0  South             3
13          2.0            250.0  South             3
14          2.0            250.0  South             3
25          2.0            250.0  South             3
1           2.0            250.0  South             3
26          2.0            250.0  South             3
27          3.0              NaN   East             4
15          3.0              NaN   East             4
3           3.0              NaN   East             4
16          4.0            450.0   West             2
4           4.0            450.0   West             2
28          4.0            450.0   West             2
6           5.0            120.0   None             7
18        

In [4]:
print("\n" + "=" * 50)
print("OUTLIER DETECTION (IQR Method)")
print("=" * 50)

def find_outliers(column):
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return column[(column < lower) | (column > upper)]

print(f"\nOutliers in 'purchase_amount': {len(find_outliers(df['purchase_amount']))}")
print(find_outliers(df['purchase_amount']).values)

print(f"\nOutliers in 'days_to_ship': {len(find_outliers(df['days_to_ship']))}")
print(find_outliers(df['days_to_ship']).values)


OUTLIER DETECTION (IQR Method)

Outliers in 'purchase_amount': 3
[5000. 5000. 5000.]

Outliers in 'days_to_ship': 3
[180 180 180]


In [5]:
print("\n" + "=" * 50)
print("CLEANING STRATEGY")
print("=" * 50)

df_clean = df.copy()

# 1. Drop rows with missing customer_id (can't identify customer)
df_clean = df_clean.dropna(subset=['customer_id'])
print(f"\nAfter dropping null customer_id: {df_clean.shape[0]} rows")

# 2. Fill missing region with 'Unknown'
df_clean['region'].fillna('Unknown', inplace=True)
print(f"Filled missing region. Nulls now: {df_clean['region'].isnull().sum()}")

# 3. Fill missing purchase_amount with median (better than mean for outliers)
median_purchase = df_clean['purchase_amount'].median()
df_clean['purchase_amount'].fillna(median_purchase, inplace=True)
print(f"Filled missing purchase_amount with median {median_purchase}")

# 4. Remove exact duplicate rows
df_clean = df_clean.drop_duplicates()
print(f"\nAfter removing duplicates: {df_clean.shape[0]} rows")

# 5. Remove shipping outliers (>100 days = data error)
df_clean = df_clean[df_clean['days_to_ship'] <= 100]
print(f"After removing extreme shipping delays: {df_clean.shape[0]} rows")

# 6. Remove purchase amount outliers (>1000 = suspicious)
df_clean = df_clean[df_clean['purchase_amount'] <= 1000]
print(f"After removing extreme purchases: {df_clean.shape[0]} rows")

print("\n" + "=" * 50)
print("CLEANED DATA")
print("=" * 50)
print(df_clean.head(10))
print(f"\nFinal shape: {df_clean.shape}")
print(f"\nMissing values:\n{df_clean.isnull().sum()}")


CLEANING STRATEGY

After dropping null customer_id: 33 rows
Filled missing region. Nulls now: 0
Filled missing purchase_amount with median 210.0

After removing duplicates: 10 rows
After removing extreme shipping delays: 9 rows
After removing extreme purchases: 9 rows

CLEANED DATA
    customer_id  purchase_amount   region  days_to_ship
0           1.0            100.0    North             5
1           2.0            250.0    South             3
3           3.0            210.0     East             4
4           4.0            450.0     West             2
6           5.0            120.0  Unknown             7
7           6.0            999.0     East             8
8           7.0            150.0    South             4
9           8.0            160.0     West             5
10          9.0            170.0    North             3

Final shape: (9, 4)

Missing values:
customer_id        0
purchase_amount    0
region             0
days_to_ship       0
dtype: int64


/tmp/ipykernel_5777/1068195642.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['region'].fillna('Unknown', inplace=True)
/tmp/ipykernel_5777/1068195642.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

In [6]:
print("\n" + "=" * 50)
print("CLEANING STRATEGY")
print("=" * 50)

df_clean = df.copy()

# 1. Drop rows with missing customer_id (can't identify customer)
df_clean = df_clean.dropna(subset=['customer_id'])
print(f"\nAfter dropping null customer_id: {df_clean.shape[0]} rows")

# 2. Fill missing region with 'Unknown'
df_clean['region'].fillna('Unknown', inplace=True)
print(f"Filled missing region. Nulls now: {df_clean['region'].isnull().sum()}")

# 3. Fill missing purchase_amount with median (better than mean for outliers)
median_purchase = df_clean['purchase_amount'].median()
df_clean['purchase_amount'].fillna(median_purchase, inplace=True)
print(f"Filled missing purchase_amount with median {median_purchase}")

# 4. Remove exact duplicate rows
df_clean = df_clean.drop_duplicates()
print(f"\nAfter removing duplicates: {df_clean.shape[0]} rows")

# 5. Remove shipping outliers (>100 days = data error)
df_clean = df_clean[df_clean['days_to_ship'] <= 100]
print(f"After removing extreme shipping delays: {df_clean.shape[0]} rows")

# 6. Remove purchase amount outliers (>1000 = suspicious)
df_clean = df_clean[df_clean['purchase_amount'] <= 1000]
print(f"After removing extreme purchases: {df_clean.shape[0]} rows")

print("\n" + "=" * 50)
print("CLEANED DATA")
print("=" * 50)
print(df_clean.head(10))
print(f"\nFinal shape: {df_clean.shape}")
print(f"\nMissing values:\n{df_clean.isnull().sum()}")


CLEANING STRATEGY

After dropping null customer_id: 33 rows
Filled missing region. Nulls now: 0
Filled missing purchase_amount with median 210.0

After removing duplicates: 10 rows
After removing extreme shipping delays: 9 rows
After removing extreme purchases: 9 rows

CLEANED DATA
    customer_id  purchase_amount   region  days_to_ship
0           1.0            100.0    North             5
1           2.0            250.0    South             3
3           3.0            210.0     East             4
4           4.0            450.0     West             2
6           5.0            120.0  Unknown             7
7           6.0            999.0     East             8
8           7.0            150.0    South             4
9           8.0            160.0     West             5
10          9.0            170.0    North             3

Final shape: (9, 4)

Missing values:
customer_id        0
purchase_amount    0
region             0
days_to_ship       0
dtype: int64


/tmp/ipykernel_5777/1068195642.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['region'].fillna('Unknown', inplace=True)
/tmp/ipykernel_5777/1068195642.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

In [7]:
print("\n" + "=" * 50)
print("BEFORE vs AFTER")
print("=" * 50)
print(f"Original rows: {df.shape[0]} → Cleaned rows: {df_clean.shape[0]}")
print(f"\nOriginal missing: {df.isnull().sum().sum()} → Cleaned missing: {df_clean.isnull().sum().sum()}")

print("\nStatistics comparison:")
print("\nBEFORE:")
print(df[['purchase_amount', 'days_to_ship']].describe())
print("\nAFTER:")
print(df_clean[['purchase_amount', 'days_to_ship']].describe())


BEFORE vs AFTER
Original rows: 36 → Cleaned rows: 9

Original missing: 9 → Cleaned missing: 0

Statistics comparison:

BEFORE:
       purchase_amount  days_to_ship
count        33.000000     36.000000
mean        740.818182     19.166667
std        1391.041886     49.211207
min         100.000000      2.000000
25%         150.000000      3.000000
50%         250.000000      4.500000
75%         500.000000      6.250000
max        5000.000000    180.000000

AFTER:
       purchase_amount  days_to_ship
count         9.000000      9.000000
mean        289.888889      4.555556
std         285.521647      1.943651
min         100.000000      2.000000
25%         150.000000      3.000000
50%         170.000000      4.000000
75%         250.000000      5.000000
max         999.000000      8.000000
